In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.filters.hp_filter import hpfilter

df = pd.read_csv(
    "../data/processed/macro_data_processed.csv",
    index_col="date",
    parse_dates=True
)

df.head()

,gdp_real,unemployment,cpi,fed_funds_rate,recession
date,,,,,
1970-01-01,5300.652,3.9,37.9,8.98,1.0
1970-02-01,NaN,4.2,38.1,8.98,1.0
1970-03-01,NaN,4.4,38.3,7.76,1.0
1970-04-01,5308.164,4.6,38.5,8.10,1.0
1970-05-01,NaN,4.8,38.6,7.95,1.0


In [2]:
gdp = df["gdp_real"].dropna()

gdp_cycle, gdp_trend = hpfilter(gdp, lamb=1600)

gdp_decomp = pd.DataFrame({
    "gdp_real": gdp,
    "trend": gdp_trend,
    "cycle": gdp_cycle,
}, index=gdp.index)

gdp_decomp.head()

c:\Users\Usuario\anaconda3\Lib\site-packages\statsmodels\tsa\filters\hp_filter.py:100: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  trend = spsolve(I+lamb*K.T.dot(K), x, use_umfpack=use_umfpack)


,gdp_real,trend,cycle
date,,,
1970-01-01,5300.652,5267.328663,33.323337
1970-04-01,5308.164,5316.979398,-8.815398
1970-07-01,5357.077,5366.650959,-9.573959
1970-10-01,5299.672,5416.358666,-116.686666
1971-01-01,5443.619,5466.111850,-22.492850


In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

recessions = df["recession"].dropna()
rec = recessions.copy()
starts = rec[(rec == 1) & (rec.shift(1) == 0)].index.tolist()
ends = rec[(rec == 0) & (rec.shift(1) == 1)].index.tolist()
if rec.iloc[0] == 1:
    starts = [rec.index[0]] + starts
pairs = list(zip(starts, ends[:len(starts)]))

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Real GDP vs Trend", "Business Cycle Component"),
    vertical_spacing=0.12
)

# GDP real vs tendência
fig.add_trace(
    go.Scatter(x=gdp_decomp.index, y=gdp_decomp["gdp_real"],
               mode="lines", name="Real GDP",
               line=dict(color="#e05c5c", width=1.5)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=gdp_decomp.index, y=gdp_decomp["trend"],
               mode="lines", name="Trend",
               line=dict(color="white", width=1.5, dash="dash")),
    row=1, col=1
)

# ciclo
fig.add_trace(
    go.Scatter(x=gdp_decomp.index, y=gdp_decomp["cycle"],
               mode="lines", name="Cycle",
               line=dict(color="#e05c5c", width=1.5),
               fill="tozeroy", fillcolor="rgba(224, 92, 92, 0.2)"),
    row=2, col=1
)

# linha zero no ciclo
fig.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.5, row=2, col=1)

# recessões
for start, end in pairs:
    for row in range(1, 3):
        fig.add_vrect(
            x0=start, x1=end,
            fillcolor="gray", opacity=0.2,
            layer="below", line_width=0,
            row=row, col=1
        )

fig.update_layout(
    height=700,
    title_text="US Business Cycles — HP Filter (λ=1600)",
    title_font_size=14,
    paper_bgcolor="#111111",
    plot_bgcolor="#111111",
    font=dict(color="white"),
    legend=dict(orientation="h", y=1.05),
)

fig.update_xaxes(gridcolor="#333333")
fig.update_yaxes(gridcolor="#333333")

fig.show()
fig.write_html("../outputs/figures/02_business_cycles.html")

In [4]:
gdp_decomp.to_csv("../data/processed/business_cycles.csv")
print("Saved.")

Saved.
